In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from pathlib import Path
from joblib import Parallel, delayed
from IPython.display import display

from joblib import Parallel, delayed, dump

from pd_estim_A.data.data_import import (
    load_data,
    load_ecb_1y_yield,
    fill_liabilities,
    drop_high_leverage_firms,
    prepare_nig_inputs,
)
from pd_estim_A.models.nig.gibbs_nig_afonso import (
    process_one_firm_bayesian_nig,
)
from pd_estim_A.data.cds_df import get_cds_panel


In [20]:
# Data processing and df preparation
print(Path.cwd())

data_path = Path.cwd() / ".." / "data" / "raw"
output_path = Path.cwd() / ".." / "data" / "derived"

# raw accounting / equity data
ret_daily, bs, coverage = load_data(
    data_path / "Jan2025_Accenture_Dataset_ErasmusCase.xlsx",
    start_date="2012-01-01",
    end_date="2025-12-19",
    enforce_coverage=True,
    coverage_tol=0.995,
    liabilities_scale="auto",
    verbose=True,
)

# risk-free rate
df_rf = load_ecb_1y_yield(
    startPeriod="2010-01-01",
    endPeriod="2025-12-31",
    out_file=output_path / "ecb_yc_1y_aaa.xml",
    verify_ssl=True,
)

# calendar and debt interpolation
df_cal = ret_daily[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)
debt_daily = fill_liabilities(bs, df_cal)

# leverage filter
ret_filt, bs_filt, lev_by_firm, dropped = drop_high_leverage_firms(
    ret_daily,
    bs,
    df_calendar=df_cal,
    debt_daily=debt_daily,
    lev_threshold=8.0,
    lev_agg="median",
    verbose=True,
)

# keep debt panel aligned with surviving firms
keep = set(ret_filt["gvkey"].astype(str).unique())
debt_daily_filt = debt_daily[debt_daily["gvkey"].astype(str).isin(keep)].copy()

# structural-model input panel for NIG
nig_out = prepare_nig_inputs(
    ret_filt,
    bs_filt,
    df_rf,
    debt_daily=debt_daily_filt,
)

# robustly extract the main DataFrame
if isinstance(nig_out, pd.DataFrame):
    nig_raw = nig_out.copy()
elif isinstance(nig_out, tuple):
    df_candidates = [x for x in nig_out if isinstance(x, pd.DataFrame)]
    if len(df_candidates) == 0:
        raise TypeError(
            "prepare_nig_inputs returned a tuple, but none of its elements is a DataFrame."
        )
    nig_raw = df_candidates[0].copy()
else:
    raise TypeError(
        f"prepare_nig_inputs returned unsupported type: {type(nig_out)}"
    )


# CDS panel
cds = get_cds_panel(
    project_root=Path.cwd() / "..",
    save_csv=False,
    verbose=True,
)

# harmonize types
nig = nig_raw.copy()
nig["gvkey"] = nig["gvkey"].astype(str)
nig["date"] = pd.to_datetime(nig["date"])

cds["gvkey"] = cds["gvkey"].astype(str)
cds["date"] = pd.to_datetime(cds["date"])

# keep only firms that appear in both datasets
common_gv = sorted(set(nig["gvkey"].unique()) & set(cds["gvkey"].unique()))
nig = nig[nig["gvkey"].isin(common_gv)].copy()
cds = cds[cds["gvkey"].isin(common_gv)].copy()

# restrict CDS to structural-panel date range
dmin, dmax = nig["date"].min(), nig["date"].max()
cds = cds[(cds["date"] >= dmin) & (cds["date"] <= dmax)].copy()

# merge CDS backward onto structural panel
nig = nig.sort_values(["date", "gvkey"]).reset_index(drop=True)
cds = cds.sort_values(["date", "gvkey"]).reset_index(drop=True)

merged_cds = pd.merge_asof(
    nig,
    cds,
    on="date",
    by="gvkey",
    direction="backward",
    allow_exact_matches=True,
)

# drop rows still missing CDS
nig_df = merged_cds.dropna(subset=["cds"]).reset_index(drop=True)

# normalize key column names to match the Bayesian-NIG module defaults
rename_map = {}
if "E" in nig_df.columns and "market_cap" not in nig_df.columns:
    rename_map["E"] = "market_cap"
if "B" in nig_df.columns and "debt_face" not in nig_df.columns:
    rename_map["B"] = "debt_face"
if "r" in nig_df.columns and "rf" not in nig_df.columns:
    rename_map["r"] = "rf"

nig_df = nig_df.rename(columns=rename_map)

print("firms after intersection:", nig_df["gvkey"].nunique())
print("rows after merge:", len(nig_df))
print("date range:", nig_df["date"].min(), "→", nig_df["date"].max())
print("columns available:", sorted(nig_df.columns))


c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test
[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
Data has been written to c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\ecb_yc_1y_aaa.xml
[drop_high_leverage_firms] agg=median, threshold=8.0
[drop_high_leverage_firms] firms before: 46 | after: 36
[drop_high_leverage_firms] dropped firms: 10
[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
[get_cds_panel] sheets read: 22 | rows parsed: 67015 | unmapped sheets: 0
firms after intersection: 21
rows after merge: 65569


In [21]:
# load precomputed frequentist NIG results for Bayesian initialization
# source file is a WEEKLY OOS panel, so we collapse it to one row per firm-window

freq_file = output_path / "nig_weekly_freq.csv"

if not freq_file.exists():
    raise FileNotFoundError(f"Could not find frequentist NIG file: {freq_file}")

freq_raw = pd.read_csv(freq_file)

# harmonize basic types
freq_raw["gvkey"] = freq_raw["gvkey"].astype(str)

date_cols = [
    "date",
    "window_train_start",
    "window_train_end",
    "window_oos_start",
    "window_oos_end",
]
for c in date_cols:
    if c in freq_raw.columns:
        freq_raw[c] = pd.to_datetime(freq_raw[c], errors="coerce")

num_cols = ["window_idx", "alpha", "beta1", "delta", "beta0"]
for c in num_cols:
    if c in freq_raw.columns:
        freq_raw[c] = pd.to_numeric(freq_raw[c], errors="coerce")

# keep only successful frequentist rows
if "ok" in freq_raw.columns:
    ok_mask = freq_raw["ok"].astype(str).str.lower().isin(["true", "1", "yes"])
    freq_raw = freq_raw.loc[ok_mask].copy()

required_cols = [
    "gvkey",
    "window_idx",
    "window_train_start",
    "window_train_end",
    "window_oos_start",
    "window_oos_end",
    "alpha",
    "beta1",
    "delta",
    "beta0",
]
missing = [c for c in required_cols if c not in freq_raw.columns]
if missing:
    raise ValueError(
        f"Missing required columns in nig_weekly_freq.csv: {missing}\n"
        f"Available columns: {list(freq_raw.columns)}"
    )

# sort first so 'last' is deterministic inside each window
freq_raw = freq_raw.sort_values(["gvkey", "window_idx", "date"]).reset_index(drop=True)

# collapse weekly OOS rows -> one row per firm-window
freq_init_df = (
    freq_raw.groupby(
        ["gvkey", "window_idx", "window_train_start", "window_train_end"],
        as_index=False,
        dropna=False,
    )
    .agg(
        train_start=("window_train_start", "first"),
        train_end=("window_train_end", "first"),
        oos_start=("window_oos_start", "first"),
        oos_end=("window_oos_end", "first"),
        alpha=("alpha", "last"),
        beta1=("beta1", "last"),
        delta=("delta", "last"),
        beta0=("beta0", "last"),
    )
)

# keep only complete usable rows
freq_init_df = freq_init_df.dropna(
    subset=["gvkey", "train_start", "train_end", "alpha", "beta1", "delta", "beta0"]
).copy()

freq_init_df = freq_init_df.sort_values(["gvkey", "train_end", "window_idx"]).reset_index(drop=True)

print("Using frequentist init file:", freq_file.name)
print("raw weekly rows:", len(freq_raw))
print("collapsed firm-window rows:", len(freq_init_df))
print("firms in init table:", freq_init_df["gvkey"].nunique())
print(
    "unique firm-window endpoints:",
    freq_init_df[["gvkey", "train_end"]].drop_duplicates().shape[0],
)

display(freq_init_df.head())


Using frequentist init file: nig_weekly_freq.csv
raw weekly rows: 10623
collapsed firm-window rows: 814
firms in init table: 21
unique firm-window endpoints: 814


,gvkey,window_idx,window_train_start,window_train_end,train_start,train_end,oos_start,oos_end,alpha,beta1,delta,beta0
0,100022,0,2012-04-06,2014-03-28,2012-04-06,2014-03-28,2014-04-04,2014-06-27,271.811197,60.985402,1.425752,-0.274974
1,100022,1,2012-07-06,2014-06-27,2012-07-06,2014-06-27,2014-07-04,2014-09-26,475.107602,239.733287,1.603545,-0.862584
2,100022,2,2012-10-05,2014-09-26,2012-10-05,2014-09-26,2014-10-03,2014-12-26,288.152866,75.712857,1.416519,-0.333768
3,100022,3,2013-01-04,2014-12-26,2013-01-04,2014-12-26,2015-01-02,2015-03-27,297.622694,67.456628,1.590442,-0.339679
4,100022,4,2013-04-05,2015-03-27,2013-04-05,2015-03-27,2015-04-03,2015-06-26,248.905592,43.254451,1.296466,-0.138839


In [22]:
# Rolling configuration
TRAIN_YEARS = 2
STEP_FREQ = "QE"
WEEK_FREQ = "W-FRI"

FORECAST_HORIZON_YEARS = 1.0
PD_HORIZON_YEARS = 1.0
ANN_FACTOR = 52.0

DATA_END = pd.Timestamp("2024-12-31")
LAST_TRAIN_END_CAL = DATA_END - pd.offsets.QuarterEnd(1)

# debug / runtime controls
MAX_FIRMS = 2
MAX_WINDOWS = 4

# Gibbs sampling controls
MAX_ITER = 2000
BURN_IN = 500
THIN = 5
SEED = 123

# default prior-dispersion controls around the frequentist NIG estimates
DEFAULT_B0_DIAG = (50.0, 50.0)
PHI_PRIOR_VARIANCE = 100.0
OMEGA = 1e-3


def next_friday(ts):
    ts = pd.Timestamp(ts)
    days_ahead = (4 - ts.weekday()) % 7
    return ts + pd.Timedelta(days=days_ahead)


def prev_friday(ts):
    ts = pd.Timestamp(ts)
    days_back = (ts.weekday() - 4) % 7
    return ts - pd.Timedelta(days=days_back)


# one-time preprocessing for speed
panel = nig_df.copy()
panel["gvkey"] = panel["gvkey"].astype(str)
panel["date"] = pd.to_datetime(panel["date"])

needed_cols = ["gvkey", "date", "company", "market_cap", "L", "rf", "cds"]
panel = panel[[c for c in needed_cols if c in panel.columns]].copy()

for c in ["market_cap", "L", "rf", "cds"]:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel = (
    panel.dropna(subset=["date", "market_cap", "L", "rf"])
         .query("market_cap > 0 and L > 0")
         .sort_values(["gvkey", "date"])
         .reset_index(drop=True)
)

# build per-firm daily panels once
firm_daily = {}
for gvkey, g in panel.groupby("gvkey", sort=False):
    g = (
        g.sort_values("date")
         .groupby("date", as_index=False)
         .last()
         .reset_index(drop=True)
    )
    firm_daily[gvkey] = g

gvkeys_all = sorted(firm_daily.keys())
if MAX_FIRMS is not None:
    gvkeys_all = gvkeys_all[:int(MAX_FIRMS)]

print("Firms loaded:", len(firm_daily), "| Firms in run:", len(gvkeys_all))
print("Panel date range:", panel["date"].min().date(), "to", panel["date"].max().date())

# build the rolling quarter schedule, then snap boundaries to Fridays
global_min_date = panel["date"].min()

earliest_end_cal = global_min_date + pd.DateOffset(years=TRAIN_YEARS) - pd.Timedelta(days=1)
train_end_cals = pd.date_range(start=earliest_end_cal, end=LAST_TRAIN_END_CAL, freq=STEP_FREQ)
train_end_cals = pd.to_datetime(train_end_cals)

if MAX_WINDOWS is not None:
    train_end_cals = train_end_cals[:int(MAX_WINDOWS)]

windows = []
for train_end_cal in train_end_cals:
    train_start_cal = train_end_cal - pd.DateOffset(years=TRAIN_YEARS) + pd.Timedelta(days=1)
    oos_start_cal = train_end_cal + pd.Timedelta(days=1)
    oos_end_cal = train_end_cal + pd.offsets.QuarterEnd(1)

    train_start = next_friday(train_start_cal)
    train_end = prev_friday(train_end_cal)
    oos_start = next_friday(oos_start_cal)
    oos_end = prev_friday(oos_end_cal)

    windows.append(
        {
            "window_idx": len(windows),
            "train_start": pd.Timestamp(train_start),
            "train_end": pd.Timestamp(train_end),
            "oos_start": pd.Timestamp(oos_start),
            "oos_end": pd.Timestamp(oos_end),
        }
    )

windows_df = pd.DataFrame(windows)

print("LAST_TRAIN_END calendar:", LAST_TRAIN_END_CAL.date())
print("Number of windows:", len(windows_df))
display(windows_df.head())


Firms loaded: 21 | Firms in run: 2
Panel date range: 2014-01-01 to 2025-12-19
LAST_TRAIN_END calendar: 2024-09-30
Number of windows: 4


,window_idx,train_start,train_end,oos_start,oos_end
0,0,2014-01-03,2015-12-25,2016-01-01,2016-03-25
1,1,2014-04-04,2016-03-25,2016-04-01,2016-06-24
2,2,2014-07-04,2016-06-24,2016-07-01,2016-09-30
3,3,2014-10-03,2016-09-30,2016-10-07,2016-12-30


In [24]:
# Quick validation run: 1 firm x 1 window
# Pick a firm-window pair that is guaranteed to exist in the frequentist init table.

VALID_MAX_ITER = 20
VALID_BURN_IN = 10
VALID_THIN = 1

test_row = (
    freq_init_df[["gvkey", "train_start", "train_end", "oos_start", "oos_end", "window_idx"]]
    .dropna()
    .sort_values(["gvkey", "train_end", "window_idx"])
    .iloc[0]
)

gvkey_test = str(test_row["gvkey"])

window_test_df = pd.DataFrame(
    [{
        "window_idx": int(test_row["window_idx"]),
        "train_start": pd.Timestamp(test_row["train_start"]),
        "train_end": pd.Timestamp(test_row["train_end"]),
        "oos_start": pd.Timestamp(test_row["oos_start"]),
        "oos_end": pd.Timestamp(test_row["oos_end"]),
    }]
)

print("gvkey_test:", gvkey_test)
display(window_test_df)

summary_test, train_test, oos_test, posterior_test = process_one_firm_bayesian_nig(
    firm_daily[gvkey_test],
    window_test_df,
    em_params_source=freq_init_df,
    gvkey_col="gvkey",
    input_frequency="daily",
    week_freq=WEEK_FREQ,
    date_col="date",
    equity_col="market_cap",
    debt_col="L",
    rf_col="rf",
    ann_factor=ANN_FACTOR,
    forecast_horizon_years=FORECAST_HORIZON_YEARS,
    pd_horizon_years=PD_HORIZON_YEARS,
    max_iter=VALID_MAX_ITER,
    burn_in=VALID_BURN_IN,
    thin=VALID_THIN,
    default_B0_diag=DEFAULT_B0_DIAG,
    phi_prior_variance=PHI_PRIOR_VARIANCE,
    omega=OMEGA,
    discounting="continuous",
    rng_seed=SEED,
)

print("summary_test shape:", summary_test.shape)
print("train_test shape:", train_test.shape)
print("oos_test shape:", oos_test.shape)
print("posterior_test objects:", len(posterior_test))

display(summary_test)
display(train_test.head())
display(oos_test.head())

gvkey_test: 100022


,window_idx,train_start,train_end,oos_start,oos_end
0,0,2012-04-06,2014-03-28,2014-04-04,2014-06-27


summary_test shape: (1, 36)
train_test shape: (13, 26)
oos_test shape: (13, 26)
posterior_test objects: 1


,gvkey,window_idx,train_start,train_end,oos_start,oos_end,train_start_req,train_end_req,oos_start_req,oos_end_req,...,delta_post_median,beta0_post_median,A_train_last_mean,A_train_last_median,n_keep_requested,n_keep_actual,n_reject,n_fail_inversion,n_fail_z,n_fail_candidate
0,100022,0,2012-04-06,2014-03-28,2014-04-04,2014-06-27,2012-04-06,2014-03-28,2014-04-04,2014-06-27,...,0.729494,-0.178224,1.581424e+11,1.581424e+11,10,10,9,0,0,9


,date,market_cap,debt_face,rf,A_train_mean,A_train_median,A_train_q05,A_train_q95,theta_train_mean,theta_train_median,...,delta_post_mean,beta0_post_mean,alpha_post_median,beta1_post_median,delta_post_median,beta0_post_median,gvkey,window_train_start,window_train_end,window_idx
0,2014-01-03,5.055556e+10,1.014480e+11,0.000991,1.531553e+11,1.531553e+11,1.531553e+11,1.531553e+11,-0.659192,-0.549929,...,0.804251,-0.306883,1.031303,0.624403,0.729494,-0.178224,100022,2012-04-06,2014-03-28,0
1,2014-01-10,5.005590e+10,1.014480e+11,0.000963,1.526611e+11,1.526611e+11,1.526611e+11,1.526611e+11,-0.659211,-0.549936,...,0.804251,-0.306883,1.031303,0.624403,0.729494,-0.178224,100022,2012-04-06,2014-03-28,0
2,2014-01-17,5.189801e+10,1.014480e+11,0.000996,1.545011e+11,1.545011e+11,1.545011e+11,1.545011e+11,-0.659189,-0.549928,...,0.804251,-0.306883,1.031303,0.624403,0.729494,-0.178224,100022,2012-04-06,2014-03-28,0
3,2014-01-24,4.934555e+10,1.014480e+11,0.000794,1.519749e+11,1.519749e+11,1.519749e+11,1.519749e+11,-0.659324,-0.549979,...,0.804251,-0.306883,1.031303,0.624403,0.729494,-0.178224,100022,2012-04-06,2014-03-28,0
4,2014-01-31,4.865325e+10,1.014480e+11,0.000371,8.367427e+12,8.367427e+12,8.367427e+12,8.367427e+12,-0.659606,-0.550087,...,0.804251,-0.306883,1.031303,0.624403,0.729494,-0.178224,100022,2012-04-06,2014-03-28,0


,date,market_cap,debt_face,rf,A_oos_mean,A_oos_median,A_oos_q05,A_oos_q95,theta_oos_mean,theta_oos_median,...,PD_Q_mean,PD_Q_median,PD_Q_q05,PD_Q_q95,gvkey,window_train_start,window_train_end,window_oos_start,window_oos_end,window_idx
0,2014-04-04,5.649725e+10,1.027250e+11,0.001217,1.245634e+11,1.269951e+11,1.084602e+11,1.475359e+11,-0.659041,-0.549872,...,0.647207,0.708494,0.326595,0.761800,100022,2012-04-06,2014-03-28,2014-04-04,2014-06-27,0
1,2014-04-11,5.430599e+10,1.027250e+11,0.001092,1.215766e+11,1.237603e+11,1.055133e+11,1.449581e+11,-0.659125,-0.549904,...,0.658634,0.721551,0.338952,0.770366,100022,2012-04-06,2014-03-28,2014-04-04,2014-06-27,0
2,2014-04-18,5.545580e+10,1.027250e+11,0.000972,1.231620e+11,1.254812e+11,1.070722e+11,1.463280e+11,-0.659204,-0.549934,...,0.652587,0.714639,0.332391,0.765850,100022,2012-04-06,2014-03-28,2014-04-04,2014-06-27,0
3,2014-04-25,5.391469e+10,1.027250e+11,0.001185,1.210308e+11,1.231666e+11,1.049784e+11,1.444863e+11,-0.659063,-0.549880,...,0.660714,0.723921,0.341231,0.771914,100022,2012-04-06,2014-03-28,2014-04-04,2014-06-27,0
4,2014-05-02,5.359563e+10,1.027250e+11,0.000924,1.206087e+11,1.227122e+11,1.045566e+11,1.441261e+11,-0.659237,-0.549946,...,0.662380,0.725763,0.343142,0.773176,100022,2012-04-06,2014-03-28,2014-04-04,2014-06-27,0


In [25]:
# Full run: all firms x eligible windows, parallelized across firms

import os

# prevent BLAS oversubscription when joblib parallelizes across firms
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

N_CORES_AVAILABLE = os.cpu_count() or 1
N_JOBS = min(len(gvkeys_all), N_CORES_AVAILABLE)

print("Available CPU cores:", N_CORES_AVAILABLE)
print("Parallel jobs:", N_JOBS)


def _stable_firm_seed(base_seed: int, gvkey: str) -> int:
    s = str(gvkey)
    offset = sum((i + 1) * ord(ch) for i, ch in enumerate(s))
    return int((base_seed + offset) % (2**32 - 1))


def _run_one_firm_bayes_nig(gvkey: str):
    gvkey = str(gvkey)

    try:
        firm_df = firm_daily[gvkey].copy()

        em_firm = (
            freq_init_df.loc[freq_init_df["gvkey"].astype(str) == gvkey].copy()
            .sort_values(["train_end", "window_idx"])
            .reset_index(drop=True)
        )

        if em_firm.empty:
            summary_fail = pd.DataFrame(
                [{
                    "gvkey": gvkey,
                    "window_idx": np.nan,
                    "train_start": pd.NaT,
                    "train_end": pd.NaT,
                    "oos_start": pd.NaT,
                    "oos_end": pd.NaT,
                    "ok": False,
                    "msg": "no_frequentist_init_for_firm",
                    "n_keep_requested": 0,
                    "n_keep_actual": 0,
                    "n_reject": 0,
                }]
            )
            return summary_fail, pd.DataFrame(), pd.DataFrame(), []

        # keep only windows that exist both in the global schedule and in the freq init table
        window_plan_firm = (
            windows_df.merge(
                em_firm[["train_start", "train_end"]].drop_duplicates(),
                on=["train_start", "train_end"],
                how="inner",
            )
            .sort_values("window_idx")
            .reset_index(drop=True)
        )

        if window_plan_firm.empty:
            summary_fail = pd.DataFrame(
                [{
                    "gvkey": gvkey,
                    "window_idx": np.nan,
                    "train_start": pd.NaT,
                    "train_end": pd.NaT,
                    "oos_start": pd.NaT,
                    "oos_end": pd.NaT,
                    "ok": False,
                    "msg": "no_matching_windows_between_schedule_and_freq_init",
                    "n_keep_requested": 0,
                    "n_keep_actual": 0,
                    "n_reject": 0,
                }]
            )
            return summary_fail, pd.DataFrame(), pd.DataFrame(), []

        firm_seed = _stable_firm_seed(SEED, gvkey)

        summary_df_i, train_df_i, oos_df_i, posterior_i = process_one_firm_bayesian_nig(
            firm_df,
            window_plan_firm,
            em_params_source=freq_init_df,
            train_start_col="train_start",
            train_end_col="train_end",
            oos_start_col="oos_start",
            oos_end_col="oos_end",
            gvkey_col="gvkey",
            input_frequency="daily",
            week_freq=WEEK_FREQ,
            date_col="date",
            equity_col="market_cap",
            debt_col="L",
            rf_col="rf",
            ann_factor=ANN_FACTOR,
            forecast_horizon_years=FORECAST_HORIZON_YEARS,
            pd_horizon_years=PD_HORIZON_YEARS,
            max_iter=MAX_ITER,
            burn_in=BURN_IN,
            thin=THIN,
            prior_b0=None,
            prior_B0=None,
            prior_hyper=None,
            default_B0_diag=DEFAULT_B0_DIAG,
            phi_prior_variance=PHI_PRIOR_VARIANCE,
            omega=OMEGA,
            discounting="continuous",
            rng_seed=firm_seed,
        )

        # relabel local window indices back to the global window_idx from windows_df
        global_map = (
            window_plan_firm[["window_idx", "train_start", "train_end", "oos_start", "oos_end"]]
            .copy()
            .reset_index(drop=True)
            .reset_index()
            .rename(columns={"index": "window_idx_local", "window_idx": "window_idx_global"})
        )

        if summary_df_i is not None and not summary_df_i.empty:
            summary_df_i = summary_df_i.merge(
                global_map,
                left_on=["window_idx", "train_start", "train_end", "oos_start", "oos_end"],
                right_on=["window_idx_local", "train_start", "train_end", "oos_start", "oos_end"],
                how="left",
            )
            summary_df_i["window_idx"] = (
                summary_df_i["window_idx_global"].fillna(summary_df_i["window_idx"]).astype(float)
            )
            summary_df_i = summary_df_i.drop(columns=["window_idx_local", "window_idx_global"], errors="ignore")

        if train_df_i is not None and not train_df_i.empty:
            map_train = global_map.rename(
                columns={"train_start": "window_train_start", "train_end": "window_train_end"}
            )
            train_df_i = train_df_i.merge(
                map_train[["window_idx_local", "window_idx_global", "window_train_start", "window_train_end"]],
                left_on=["window_idx", "window_train_start", "window_train_end"],
                right_on=["window_idx_local", "window_train_start", "window_train_end"],
                how="left",
            )
            train_df_i["window_idx"] = (
                train_df_i["window_idx_global"].fillna(train_df_i["window_idx"]).astype(float)
            )
            train_df_i = train_df_i.drop(columns=["window_idx_local", "window_idx_global"], errors="ignore")

        if oos_df_i is not None and not oos_df_i.empty:
            map_oos = global_map.rename(
                columns={
                    "train_start": "window_train_start",
                    "train_end": "window_train_end",
                    "oos_start": "window_oos_start",
                    "oos_end": "window_oos_end",
                }
            )
            oos_df_i = oos_df_i.merge(
                map_oos[
                    [
                        "window_idx_local",
                        "window_idx_global",
                        "window_train_start",
                        "window_train_end",
                        "window_oos_start",
                        "window_oos_end",
                    ]
                ],
                left_on=[
                    "window_idx",
                    "window_train_start",
                    "window_train_end",
                    "window_oos_start",
                    "window_oos_end",
                ],
                right_on=[
                    "window_idx_local",
                    "window_train_start",
                    "window_train_end",
                    "window_oos_start",
                    "window_oos_end",
                ],
                how="left",
            )
            oos_df_i["window_idx"] = (
                oos_df_i["window_idx_global"].fillna(oos_df_i["window_idx"]).astype(float)
            )
            oos_df_i = oos_df_i.drop(columns=["window_idx_local", "window_idx_global"], errors="ignore")

        if posterior_i:
            key_map = {
                (
                    pd.Timestamp(r["train_start"]),
                    pd.Timestamp(r["train_end"]),
                    pd.Timestamp(r["oos_start"]),
                    pd.Timestamp(r["oos_end"]),
                ): int(r["window_idx_global"])
                for _, r in global_map.iterrows()
            }
            for p in posterior_i:
                key = (
                    pd.Timestamp(p["train_start"]),
                    pd.Timestamp(p["train_end"]),
                    pd.Timestamp(p["oos_start"]),
                    pd.Timestamp(p["oos_end"]),
                )
                if key in key_map:
                    p["window_idx"] = key_map[key]

        return summary_df_i, train_df_i, oos_df_i, posterior_i

    except Exception as exc:
        summary_fail = pd.DataFrame(
            [{
                "gvkey": gvkey,
                "window_idx": np.nan,
                "train_start": pd.NaT,
                "train_end": pd.NaT,
                "oos_start": pd.NaT,
                "oos_end": pd.NaT,
                "ok": False,
                "msg": f"firm_job_fail:{type(exc).__name__}:{str(exc)[:250]}",
                "n_keep_requested": 0,
                "n_keep_actual": 0,
                "n_reject": 0,
            }]
        )
        return summary_fail, pd.DataFrame(), pd.DataFrame(), []


parallel_results = Parallel(
    n_jobs=N_JOBS,
    backend="loky",
    verbose=10,
)(
    delayed(_run_one_firm_bayes_nig)(gvkey)
    for gvkey in gvkeys_all
)

summary_parts = []
train_parts = []
oos_parts = []
posterior_all = []

for summary_df_i, train_df_i, oos_df_i, posterior_i in parallel_results:
    if summary_df_i is not None and not summary_df_i.empty:
        summary_parts.append(summary_df_i.copy())
    if train_df_i is not None and not train_df_i.empty:
        train_parts.append(train_df_i.copy())
    if oos_df_i is not None and not oos_df_i.empty:
        oos_parts.append(oos_df_i.copy())
    if posterior_i:
        posterior_all.extend(posterior_i)

summary_all = (
    pd.concat(summary_parts, ignore_index=True)
    .sort_values(["gvkey", "window_idx", "train_end"], na_position="last")
    .reset_index(drop=True)
    if summary_parts else pd.DataFrame()
)

train_all = (
    pd.concat(train_parts, ignore_index=True)
    .sort_values(["gvkey", "window_idx", "date"], na_position="last")
    .reset_index(drop=True)
    if train_parts else pd.DataFrame()
)

oos_all = (
    pd.concat(oos_parts, ignore_index=True)
    .sort_values(["gvkey", "window_idx", "date"], na_position="last")
    .reset_index(drop=True)
    if oos_parts else pd.DataFrame()
)

print("summary_all shape:", summary_all.shape)
print("train_all shape:", train_all.shape)
print("oos_all shape:", oos_all.shape)
print("posterior objects:", len(posterior_all))

if not summary_all.empty and "ok" in summary_all.columns:
    ok_mask = summary_all["ok"].astype(str).str.lower().isin(["true", "1", "yes"])
    print("successful windows:", int(ok_mask.sum()), "/", len(summary_all))

display(summary_all.head())
display(train_all.head())
display(oos_all.head())

Available CPU cores: 12
Parallel jobs: 2


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.


summary_all shape: (8, 36)
train_all shape: (834, 26)
oos_all shape: (106, 26)
posterior objects: 8
successful windows: 8 / 8


[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed: 14.4min finished


,gvkey,window_idx,train_start,train_end,oos_start,oos_end,train_start_req,train_end_req,oos_start_req,oos_end_req,...,delta_post_median,beta0_post_median,A_train_last_mean,A_train_last_median,n_keep_requested,n_keep_actual,n_reject,n_fail_inversion,n_fail_z,n_fail_candidate
0,100022,0.0,2014-01-03,2015-12-25,2016-01-01,2016-03-25,2014-01-03,2015-12-25,2016-01-01,2016-03-25,...,3.715438,-8.024773,1.771421e+11,1.771421e+11,300,300,0,0,0,0
1,100022,1.0,2014-04-04,2016-03-25,2016-04-01,2016-06-24,2014-04-04,2016-03-25,2016-04-01,2016-06-24,...,1.158733,0.089815,1.779334e+11,1.779334e+11,300,300,0,0,0,0
2,100022,2.0,2014-07-04,2016-06-24,2016-07-01,2016-09-30,2014-07-04,2016-06-24,2016-07-01,2016-09-30,...,2.553561,-0.048994,1.715651e+11,1.715651e+11,300,300,0,0,0,0
3,100022,3.0,2014-10-03,2016-09-30,2016-10-07,2016-12-30,2014-10-03,2016-09-30,2016-10-07,2016-12-30,...,2.005126,-0.000878,1.753807e+11,1.753807e+11,300,300,0,0,0,0
4,100080,0.0,2014-01-03,2015-12-25,2016-01-01,2016-03-25,2014-01-03,2015-12-25,2016-01-01,2016-03-25,...,6.148915,14.556640,1.462162e+11,1.462162e+11,300,300,0,0,0,0


,date,market_cap,debt_face,rf,A_train_mean,A_train_median,A_train_q05,A_train_q95,theta_train_mean,theta_train_median,...,delta_post_mean,beta0_post_mean,alpha_post_median,beta1_post_median,delta_post_median,beta0_post_median,gvkey,window_train_start,window_train_end,window_idx
0,2014-01-03,5.055556e+10,1.014480e+11,0.000991,1.675173e+11,1.675173e+11,1.675173e+11,1.675173e+11,-4.085009,-4.723222,...,3.725288,-8.077794,6818.623408,6195.275657,3.715438,-8.024773,100022,2014-01-03,2015-12-25,0.0
1,2014-01-10,5.005590e+10,1.014480e+11,0.000963,1.670277e+11,1.670277e+11,1.670277e+11,1.670277e+11,-4.088820,-4.726954,...,3.725288,-8.077794,6818.623408,6195.275657,3.715438,-8.024773,100022,2014-01-03,2015-12-25,0.0
2,2014-01-17,5.189801e+10,1.014480e+11,0.000996,1.688738e+11,1.688738e+11,1.688738e+11,1.688738e+11,-4.084345,-4.722572,...,3.725288,-8.077794,6818.623408,6195.275657,3.715438,-8.024773,100022,2014-01-03,2015-12-25,0.0
3,2014-01-24,4.934555e+10,1.014480e+11,0.000794,1.663774e+11,1.663774e+11,1.663774e+11,1.663774e+11,-4.111729,-4.749390,...,3.725288,-8.077794,6818.623408,6195.275657,3.715438,-8.024773,100022,2014-01-03,2015-12-25,0.0
4,2014-01-31,4.865325e+10,1.014480e+11,0.000371,1.658266e+11,1.658266e+11,1.658266e+11,1.658266e+11,-4.168970,-4.805446,...,3.725288,-8.077794,6818.623408,6195.275657,3.715438,-8.024773,100022,2014-01-03,2015-12-25,0.0


,date,market_cap,debt_face,rf,A_oos_mean,A_oos_median,A_oos_q05,A_oos_q95,theta_oos_mean,theta_oos_median,...,PD_Q_mean,PD_Q_median,PD_Q_q05,PD_Q_q95,gvkey,window_train_start,window_train_end,window_oos_start,window_oos_end,window_idx
0,2016-01-01,5.877279e+10,1.173660e+11,-0.003968,1.766054e+11,1.766054e+11,1.766054e+11,1.766054e+11,-4.757158,-5.406781,...,0.000002,0.000001,1.142334e-07,0.000008,100022,2014-01-03,2015-12-25,2016-01-01,2016-03-25,0.0
1,2016-01-08,5.023048e+10,1.173660e+11,-0.004354,1.681086e+11,1.681086e+11,1.681085e+11,1.681086e+11,-4.809560,-5.454317,...,0.000024,0.000017,3.075359e-06,0.000080,100022,2014-01-03,2015-12-25,2016-01-01,2016-03-25,0.0
2,2016-01-15,4.696767e+10,1.173660e+11,-0.004190,1.648264e+11,1.648264e+11,1.648260e+11,1.648265e+11,-4.787315,-5.434137,...,0.000061,0.000045,1.000384e-05,0.000184,100022,2014-01-03,2015-12-25,2016-01-01,2016-03-25,0.0
3,2016-01-22,4.798504e+10,1.173660e+11,-0.004325,1.658597e+11,1.658597e+11,1.658594e+11,1.658598e+11,-4.805621,-5.450744,...,0.000046,0.000033,6.964995e-06,0.000142,100022,2014-01-03,2015-12-25,2016-01-01,2016-03-25,0.0
4,2016-01-29,4.614895e+10,1.173660e+11,-0.004508,1.640450e+11,1.640451e+11,1.640446e+11,1.640452e+11,-4.830392,-5.473214,...,0.000077,0.000058,1.342241e-05,0.000227,100022,2014-01-03,2015-12-25,2016-01-01,2016-03-25,0.0


In [26]:
# Save outputs
bayes_nig_out = output_path / "bayesian_nig"
bayes_nig_out.mkdir(parents=True, exist_ok=True)

run_tag = (
    f"condfixA_iter{MAX_ITER}_burn{BURN_IN}_thin{THIN}_"
    f"firms{len(gvkeys_all)}_wins{len(windows_df)}"
)

summary_file = bayes_nig_out / f"bayes_nig_summary_{run_tag}.csv"
train_file = bayes_nig_out / f"bayes_nig_train_weekly_{run_tag}.csv"
oos_file = bayes_nig_out / f"bayes_nig_oos_weekly_{run_tag}.csv"
windows_file = bayes_nig_out / f"bayes_nig_windows_{run_tag}.csv"
settings_file = bayes_nig_out / f"bayes_nig_run_settings_{run_tag}.csv"
posterior_file = bayes_nig_out / f"bayes_nig_posterior_{run_tag}.joblib"

if not summary_all.empty:
    summary_all.to_csv(summary_file, index=False)

if not train_all.empty:
    train_all.to_csv(train_file, index=False)

if not oos_all.empty:
    oos_all.to_csv(oos_file, index=False)

windows_df.to_csv(windows_file, index=False)

run_settings = pd.DataFrame(
    [
        {
            "train_years": TRAIN_YEARS,
            "step_freq": STEP_FREQ,
            "week_freq": WEEK_FREQ,
            "forecast_horizon_years": FORECAST_HORIZON_YEARS,
            "pd_horizon_years": PD_HORIZON_YEARS,
            "ann_factor": ANN_FACTOR,
            "data_end": DATA_END,
            "last_train_end_cal": LAST_TRAIN_END_CAL,
            "max_firms": MAX_FIRMS,
            "max_windows": MAX_WINDOWS,
            "max_iter": MAX_ITER,
            "burn_in": BURN_IN,
            "thin": THIN,
            "seed": SEED,
            "default_B0_diag_1": DEFAULT_B0_DIAG[0],
            "default_B0_diag_2": DEFAULT_B0_DIAG[1],
            "phi_prior_variance": PHI_PRIOR_VARIANCE,
            "omega": OMEGA,
            "n_jobs": N_JOBS,
            "n_firms_run": len(gvkeys_all),
            "n_windows_schedule": len(windows_df),
            "n_summary_rows": len(summary_all),
            "n_train_rows": len(train_all),
            "n_oos_rows": len(oos_all),
            "n_posterior_objects": len(posterior_all),
        }
    ]
)
run_settings.to_csv(settings_file, index=False)

dump(posterior_all, posterior_file, compress=3)

print("Saved files:")
for fp in [summary_file, train_file, oos_file, windows_file, settings_file, posterior_file]:
    print(" -", fp)

Saved files:
 - c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\bayesian_nig\bayes_nig_summary_condfixA_iter2000_burn500_thin5_firms2_wins4.csv
 - c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\bayesian_nig\bayes_nig_train_weekly_condfixA_iter2000_burn500_thin5_firms2_wins4.csv
 - c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\bayesian_nig\bayes_nig_oos_weekly_condfixA_iter2000_burn500_thin5_firms2_wins4.csv
 - c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\bayesian_nig\bayes_nig_windows_condfixA_iter2000_burn500_thin5_firms2_wins4.csv
 - c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\bayesian_nig\bayes_nig_run_settings_condfixA_iter2000_burn500_thin5_firms2_wins4.csv
 - c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\bayesian_nig\bay

In [27]:
# Quick diagnostics

if not summary_all.empty:
    cols_show = [c for c in [
        "gvkey",
        "window_idx",
        "train_start",
        "train_end",
        "oos_start",
        "oos_end",
        "ok",
        "msg",
        "n_keep_requested",
        "n_keep_actual",
        "n_reject",
        "n_fail_inversion",
        "n_fail_z",
        "n_fail_candidate",
        "alpha_post_median",
        "beta1_post_median",
        "delta_post_median",
        "beta0_post_median",
    ] if c in summary_all.columns]

    display(summary_all[cols_show].head(20))

    if "ok" in summary_all.columns:
        fail_mask = ~summary_all["ok"].astype(str).str.lower().isin(["true", "1", "yes"])
        if fail_mask.any():
            print("Most common failure messages:")
            display(summary_all.loc[fail_mask, "msg"].value_counts().head(20))

if not oos_all.empty:
    cols_show_oos = [c for c in [
        "gvkey",
        "window_idx",
        "date",
        "A_oos_median",
        "theta_oos_median",
        "PD_P_median",
        "PD_Q_median",
    ] if c in oos_all.columns]
    display(oos_all[cols_show_oos].head(20))

,gvkey,window_idx,train_start,train_end,oos_start,oos_end,ok,msg,n_keep_requested,n_keep_actual,n_reject,n_fail_inversion,n_fail_z,n_fail_candidate,alpha_post_median,beta1_post_median,delta_post_median,beta0_post_median
0,100022,0.0,2014-01-03,2015-12-25,2016-01-01,2016-03-25,True,ok,300,300,0,0,0,0,6818.623408,6195.275657,3.715438,-8.024773
1,100022,1.0,2014-04-04,2016-03-25,2016-04-01,2016-06-24,True,ok,300,300,0,0,0,0,152.530467,-14.074639,1.158733,0.089815
2,100022,2.0,2014-07-04,2016-06-24,2016-07-01,2016-09-30,True,ok,300,300,0,0,0,0,303.431672,1.143315,2.553561,-0.048994
3,100022,3.0,2014-10-03,2016-09-30,2016-10-07,2016-12-30,True,ok,300,300,0,0,0,0,248.241382,-2.092685,2.005126,-0.000878
4,100080,0.0,2014-01-03,2015-12-25,2016-01-01,2016-03-25,True,ok,300,300,0,0,0,0,3893.262433,-3593.823294,6.148915,14.556640
5,100080,1.0,2014-04-04,2016-03-25,2016-04-01,2016-06-24,True,ok,300,300,0,0,0,0,368.329225,-235.816484,5.468713,4.514728
6,100080,2.0,2014-07-04,2016-06-24,2016-07-01,2016-09-30,True,ok,300,300,0,0,0,0,9863.096274,-9689.410924,2.561009,13.439612
7,100080,3.0,2014-10-03,2016-09-30,2016-10-07,2016-12-30,True,ok,300,300,0,0,0,0,5303.967188,-5131.594742,3.037136,11.514343


,gvkey,window_idx,date,A_oos_median,theta_oos_median,PD_P_median,PD_Q_median
0,100022,0.0,2016-01-01,1.766054e+11,-5.406781,9.521656e-08,0.000001
1,100022,0.0,2016-01-08,1.681086e+11,-5.454317,1.972047e-06,0.000017
2,100022,0.0,2016-01-15,1.648264e+11,-5.434137,6.096827e-06,0.000045
3,100022,0.0,2016-01-22,1.658597e+11,-5.450744,4.289828e-06,0.000033
4,100022,0.0,2016-01-29,1.640451e+11,-5.473214,7.904030e-06,0.000058
5,100022,0.0,2016-02-05,1.614964e+11,-5.483607,1.756723e-05,0.000124
6,100022,0.0,2016-02-12,1.602329e+11,-5.515828,2.605540e-05,0.000181
7,100022,0.0,2016-02-19,1.623880e+11,-5.521895,1.328962e-05,0.000097
8,100022,0.0,2016-02-26,1.624402e+11,-5.538649,1.307346e-05,0.000096
9,100022,0.0,2016-03-04,1.677078e+11,-5.544870,2.267768e-06,0.000020
